# Setup

In [211]:
import os
import math
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict
import re


In [212]:
# Function to load in the files
def load_data(run):
    bike_frames = []
    drone_frames = []

    n_bikes = len(os.listdir(f"../logs/bike/run_{run}"))
    n_drones = len(os.listdir(f"../logs/drone/run_{run}"))

    name = ""

    for i in range(1, n_bikes + 1):
        bike_file = f"../logs/bike/run_{run}/bike_{i}.csv"
        with open(bike_file, "r") as f:
            name = f.readline().strip()
            
        if os.path.exists(bike_file):
            bike_df = pd.read_csv(bike_file, header=1, skipinitialspace=True)
            bike_df.insert(loc=0, column="ID", value=i)
            bike_frames.append(bike_df)

    for i in range(1, n_drones + 1):
        drone_file = f"../logs/drone/run_{run}/drone_{i}.csv"
        if os.path.exists(drone_file):
            drone_df = pd.read_csv(drone_file, header=1, skipinitialspace=True)
            drone_df.insert(loc=0, column="ID", value=i)
            drone_frames.append(drone_df)

    bike_data = pd.concat(bike_frames, ignore_index=True) if bike_frames else pd.DataFrame()
    drone_data = pd.concat(drone_frames, ignore_index=True) if drone_frames else pd.DataFrame()
    bike_data.name = name
    drone_data.name = name

    return bike_data, drone_data

In [213]:
def add_nearby_drones(bike_data, drone_data, radius=50):
    bike_data = bike_data.copy()

    bike_data["nearby_drones"] = 0
    bike_data["min_distance"] = 0

    for i, bike in bike_data.iterrows():
        count = 0
        min_distance = float('inf')

        same_time_drones = drone_data[drone_data["Timestep"] == bike["Timestep"]]

        for _, drone in same_time_drones.iterrows():
            if (
                abs(bike["Pos x"] - drone["Pos x"]) <= radius and
                abs(bike["Pos y"] - drone["Pos y"]) <= radius and
                abs(bike["Pos z"] - drone["Pos z"]) <= radius
            ):
                count += 1
            distance = math.sqrt(pow(bike["Pos x"] - drone["Pos x"], 2) + pow(bike["Pos y"] - drone["Pos y"], 2) + pow(bike["Pos z"] - drone["Pos z"], 2))
            min_distance = min(min_distance, distance)
        
        bike_data.at[i, "min_distance"] = min_distance
        bike_data.at[i, "nearby_drones"] = count

    return bike_data

In [214]:
def add_nearby_bikes(bike_data, drone_data, radius=50):
    drone_data = drone_data.copy()

    drone_data["nearby_bikes"] = 0
    drone_data["min_distance"] = 0

    for i, drone in drone_data.iterrows():
        count = 0
        min_distance = float('inf')

        same_time_bikes = bike_data[bike_data["Timestep"] == drone["Timestep"]]

        for _, bike in same_time_bikes.iterrows():
            if (
                abs(drone["Pos x"] - bike["Pos x"]) <= radius and
                abs(drone["Pos y"] - bike["Pos y"]) <= radius and
                abs(drone["Pos z"] - bike["Pos z"]) <= radius
            ):
                count += 1

            distance = math.sqrt(pow(bike["Pos x"] - drone["Pos x"], 2) + pow(bike["Pos y"] - drone["Pos y"], 2) + pow(bike["Pos z"] - drone["Pos z"], 2))
            min_distance = min(min_distance, distance)

        drone_data.at[i, "min_distance"] = min_distance
        drone_data.at[i, "nearby_bikes"] = count

    return drone_data

In [215]:
all_bike_data = []
all_drone_data = []

def setup():
    run = len(os.listdir("../logs/bike"))

    for i in range(1, run+1):
        bike_data, drone_data = load_data(i)
        name = bike_data.name

        bike_data = add_nearby_drones(bike_data, drone_data)
        drone_data = add_nearby_bikes(bike_data, drone_data)
        
        bike_data.name = name
        drone_data.name = name

        all_bike_data.append(bike_data)
        all_drone_data.append(drone_data)

    return all_bike_data, all_drone_data

#### Change amount of drones and bikes below to fit the numbers on the run


In [216]:
bike_data_list, drone_data_list = setup()

runs = len(bike_data_list)


In [217]:
info_dict = defaultdict(lambda: defaultdict())

for i in range(1, runs+1):
    #Save to csv for later use
    bike_data_list[i-1].to_csv(f"data/bike/run_{i}.csv", index=False)
    drone_data_list[i-1].to_csv(f"data/drone/run_{i}.csv", index=False)

    info = re.findall(r"\d+", bike_data_list[i-1].name)
    info_dict[i]['bikes'] = info[0]
    info_dict[i]['drones'] = info[1]
    info_dict[i]['stage'] = info[2]


# Plot data

#### Average distance from drone to nearest bike

In [218]:
for i in range(1, runs+1):
    df = pd.read_csv(f"data/drone/run_{i}.csv")
    avg_dist = df.groupby("Timestep")["min_distance"].agg(["mean", "min", "max"]).reset_index()

    sns.lineplot(data=avg_dist, x="Timestep", y="mean")

    plt.fill_between(
        avg_dist["Timestep"],
        avg_dist["min"],
        avg_dist["max"],
        alpha=0.4
    )

    plt.title(f"Average distance from drone to nearest bike over time \n Number of bikes: {info_dict[i]['bikes']} - Number of drones: {info_dict[i]['drones']} - Stage: {info_dict[i]['stage']}")
    plt.xlabel("Timestep")
    plt.ylabel("Average Min Distance")
    plt.show()

#### Average distance from bike to nearest drone

In [219]:
for i in range(1, runs+1):
    df = pd.read_csv(f"data/bike/run_{i}.csv")
    avg_dist = df.groupby("Timestep")["min_distance"].agg(["mean", "min", "max"]).reset_index()

    sns.lineplot(data=avg_dist, x="Timestep", y="mean")

    plt.fill_between(
        avg_dist["Timestep"],
        avg_dist["min"],
        avg_dist["max"],
        alpha=0.4
    )

    plt.title(f"Average distance from bike to nearest drone over time \n Number of bikes: {info_dict[i]['bikes']} - Number of drones: {info_dict[i]['drones']} - Stage: {info_dict[i]['stage']}")
    plt.xlabel("Timestep")
    plt.ylabel("Average Min Distance")
    plt.show()

### Percentage of bikes in range of a drone


In [220]:
for i in range(1, runs+1):
    df = pd.read_csv(f"data/bike/run_{i}.csv")
    percentage_per_time = (
        df.groupby("Timestep")["nearby_drones"]
        .apply(lambda x: (x > 0).mean() * 100)
        .reset_index(name="bikes_in_range_of_drones")
    )

    sns.lineplot(data=percentage_per_time, x="Timestep", y="bikes_in_range_of_drones")
    plt.gca().yaxis.set_major_formatter(lambda x, _: f"{x:.0f}%")
    plt.ylim(0, 101)
    plt.title(f"Percentage of bikes in range of a drone \n Number of bikes: {info_dict[i]['bikes']} - Number of drones: {info_dict[i]['drones']} - Stage: {info_dict[i]['stage']}")
    plt.ylabel("Bikes in range")
    plt.show()

### Collisions over time

In [221]:
for i in range(1, runs+1):
    df = pd.read_csv(f"data/drone/run_{i}.csv")
    collisions_at_time = df.groupby("Timestep")["Collisions"].sum().reset_index()

    #Each drone in a collision detects it, so we divide by 2
    collisions_at_time["Collisions"] = collisions_at_time["Collisions"] / 2
    collisions_at_time["Cumulative_collisions"] = collisions_at_time["Collisions"].cumsum()

    sns.lineplot(data=collisions_at_time, x="Timestep", y="Collisions")

    sns.lineplot(
        data=collisions_at_time, x="Timestep", y="Cumulative_collisions"
    )

    plt.title(f"Cumulative Collisions per Timestep Over Time \n Number of bikes: {info_dict[i]['bikes']} - Number of drones: {info_dict[i]['drones']} - Stage: {info_dict[i]['stage']}")
    plt.show()


### Average amount of bikes in range of a drone over time

In [222]:
for i in range(1, runs+1):
    df = pd.read_csv(f"data/drone/run_{i}.csv")
    nearby_bikes_stats = df.groupby("Timestep")["nearby_bikes"].agg(["mean", "min", "max"]).reset_index()

    sns.lineplot(data=nearby_bikes_stats, x="Timestep", y="mean")

    plt.fill_between(
        nearby_bikes_stats["Timestep"],
        nearby_bikes_stats["min"],
        nearby_bikes_stats["max"],
        alpha=0.4
    )

    plt.title(f"Average amount of bikes in range of any drone over time \n Number of bikes: {info_dict[i]['bikes']} - Number of drones: {info_dict[i]['drones']} - Stage: {info_dict[i]['stage']}")
    plt.ylabel("Average amount nearby bikes")
    plt.show()